# 1. What you'll learn

        - watch assignment/update steps minimize inertia
- compare elbow and silhouette evidence
- break k-means on moons and unscaled features

        **The one question this notebook answers:** What does k-means optimize, and why do scale and non-spherical shape defeat its tidy clusters?

# 2. Setup

This lesson keeps k-means clustering small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_blobs,make_moons
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score,adjusted_rand_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Three seeded spherical blobs with known labels teach the favourable case; a two-moons dataset is reserved for the failure experiment.

**Small example:** For points [0,1,2,8,9], centroids 1 and 8.5 assign the first three left and last two right; updating means returns 1 and 8.5, so the algorithm has converged.

In [ ]:
X,truth=make_blobs(n_samples=750,centers=[(-4,-1),(0,3),(4,-.5)],cluster_std=[.75,1.0,.8],random_state=SEED);frame=pd.DataFrame(X,columns=["x1","x2"]).assign(true_group=truth);print("Shape:",frame.shape,"hidden group counts:",frame.true_group.value_counts().to_dict())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(X[:,0],X[:,1],s=12,color="#176b66");axes[0].set_title("Unlabelled geometry")
axes[1].hist(X[:,0],bins=30,color="#d38b45");axes[1].set_title("Multimodal x1")
axes[2].scatter(X[:,0],X[:,1],c=truth,cmap="viridis",s=12);axes[2].set_title("Truth, evaluation only");plt.tight_layout();plt.show()

**Takeaway:** The visible groups are compact and roughly spherical—the favourable k-means geometry.

**Takeaway:** Marginal modes suggest more than one center but do not determine k alone.

**Takeaway:** Known simulation labels are hidden during fitting and used only after clustering.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
Xs=StandardScaler().fit_transform(X);ks=range(2,8);fits={k:KMeans(n_clusters=k,n_init=15,random_state=SEED).fit(Xs) for k in ks};print("Scaled shape:",Xs.shape)

# 6. Train

        Training turns the assumptions behind k-means clustering into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Seed k candidate centroids.
2. Assign each row to its nearest centroid.
3. Replace each centroid with its assigned-row mean.
4. Repeat assignment and update until inertia no longer decreases.

In [ ]:
inertia=[fits[k].inertia_ for k in ks];sil=[silhouette_score(Xs,fits[k].labels_) for k in ks];best_k=list(ks)[int(np.argmax(sil))];model=fits[best_k];print("Silhouette by k:",dict(zip(ks,np.round(sil,3))),"chosen k:",best_k)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
ari=adjusted_rand_score(truth,model.labels_);rng=np.random.default_rng(SEED);random_labels=rng.integers(0,best_k,len(X));base=adjusted_rand_score(truth,random_labels)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(list(ks),inertia,marker="o");axes[0].set(title="Elbow in inertia",xlabel="k");axes[1].plot(list(ks),sil,marker="o",color="#d38b45");axes[1].set(title="Silhouette",xlabel="k")
axes[2].scatter(X[:,0],X[:,1],c=model.labels_,cmap="viridis",s=12);cent=StandardScaler().fit(X).inverse_transform(model.cluster_centers_);axes[2].scatter(cent[:,0],cent[:,1],marker="X",s=160,color="red");axes[2].set_title(f"Chosen clusters · ARI {ari:.3f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Random-label ARI baseline={base:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Squared Euclidean distance assumes spherical equal-variance groups and is dominated by arbitrary feature units. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
moons,moon_truth=make_moons(n_samples=600,noise=.07,random_state=SEED);moon_pred=KMeans(n_clusters=2,n_init=15,random_state=SEED).fit_predict(moons)
scaled_data=X.copy();scaled_data[:,0]*=100;raw_pred=KMeans(n_clusters=3,n_init=15,random_state=SEED).fit_predict(scaled_data)
print(f"Spherical blobs ARI={ari:.3f}; moons ARI={adjusted_rand_score(moon_truth,moon_pred):.3f}; unscaled-unit blobs ARI={adjusted_rand_score(truth,raw_pred):.3f}")

**Use this when…** clusters are compact, similarly sized, and meaningful under Euclidean distance.

        **Don't use this when…** groups curve, vary greatly in density/size, contain outliers, or k has no operational meaning.

        ## Try this

        1. Run n_init=1 under several seeds and compare local optima.
2. Add outliers and watch centroids move.
3. Standardize versus robust-scale a heavy-tailed feature.